<a href="https://colab.research.google.com/github/jaalvalcan/GEE_index_sets/blob/main/LAZ_lidar_descrom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "laspy[lazrs]"

In [ ]:
import laspy
import numpy as np

# Ensure you have run: pip install "laspy[lazrs]"
file_path = '/content/sample_data/NN7232_2PPM_LAS_PHASE1.laz'

def deep_explore_lidar(path):
    try:
        # Open the file (Header only - no backend needed yet)
        with laspy.open(path) as fh:
            header = fh.header

            print("=== FILE SPECIFICATIONS ===")
            print(f"Version:            {header.version}")
            print(f"Point Format:       {header.point_format.id}")
            print(f"Generating Software: {header.generating_software}")
            print(f"System ID:          {header.system_identifier}")

            print("\n=== GEOGRAPHICAL / CRS INFO ===")
            # LAZ files store CRS in Variable Length Records (VLRs)
            if not header.vlrs:
                print("Result: No CRS records found (Common in raw Phase 1 data).")
            else:
                for vlr in header.vlrs:
                    print(f"VLR Description: {vlr.description}")
                    print(f" - User ID: {vlr.user_id}")
                    print(f" - Record ID: {vlr.record_id}")
                    # If this were a standard CRS, we would see GeoKeyDirectoryTag here

            print("\n=== BOUNDS & DENSITY ===")
            x_size = header.max[0] - header.min[0]
            y_size = header.max[1] - header.min[1]
            print(f"Area Cover: {x_size:.2f} x {y_size:.2f} units")
            print(f"Total Points: {header.point_count:,}")

            print("\n=== POINT DATA (Requires lazrs backend) ===")
            # This is where the decompression happens
            las = fh.read()

            # Classification Analysis
            classes, counts = np.unique(las.classification, return_counts=True)
            class_map = {1: "Unassigned", 2: "Ground", 3: "Low Veg",
                         4: "Med Veg", 5: "High Veg", 6: "Building", 7: "Noise"}

            for cls, count in zip(classes, counts):
                label = class_map.get(cls, "Other")
                print(f" - Class {cls} ({label}): {count:,} points")

    except Exception as e:
        print(f"\nCRITICAL ERROR: {e}")
        if "No LazBackend" in str(e):
            print("FIX: Run 'pip install \"laspy[lazrs]\"' to enable decompression.")

deep_explore_lidar(file_path)

In [ ]:
import laspy
import matplotlib.pyplot as plt
import numpy as np

file_path = '/content/sample_data/NN7232_2PPM_LAS_PHASE1.laz'

def plot_professional_histograms(path):
    # Load the data
    las = laspy.read(path)

    # Set a clean, professional style
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['axes.edgecolor'] = '#333333'
    plt.rcParams['axes.linewidth'] = 0.8

    # 1. ELEVATION (Z) HISTOGRAM
    z_data = las.z
    z_mean = np.mean(z_data)
    z_median = np.median(z_data)

    plt.figure(figsize=(5, 5))
    # Use a professional 'Steel Blue' or 'Deep Teal' color
    plt.hist(z_data, bins=120, color='#2c7bb6', edgecolor='#ffffff', linewidth=0.5, alpha=0.85)

    # Add vertical lines for Mean and Median
    plt.axvline(z_mean, color='#d7191c', linestyle='--', label=f'Mean: {z_mean:.2f}')
    plt.axvline(z_median, color='#fdae61', linestyle='-', label=f'Median: {z_median:.2f}')

    plt.title('Elevation Profile Distribution (Z-axis)', fontsize=14, pad=15, fontweight='bold', color='#212121')
    plt.xlabel('Elevation (Units)', fontsize=11, labelpad=10)
    plt.ylabel('Frequency', fontsize=11, labelpad=10)
    plt.grid(axis='y', linestyle=':', alpha=0.6)
    plt.legend(frameon=True, facecolor='white', edgecolor='none')

    # Despine (remove top and right borders)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

    # 2. INTENSITY HISTOGRAM
    if hasattr(las, 'intensity'):
        i_data = las.intensity
        plt.figure(figsize=(5, 5))
        # Use a 'Slate Gray' or 'Emerald' theme for intensity
        plt.hist(i_data, bins=100, color='#018571', edgecolor='#ffffff', linewidth=0.5, alpha=0.85)

        plt.title('Pulse Reflection Intensity Distribution', fontsize=14, pad=15, fontweight='bold', color='#212121')
        plt.xlabel('Intensity Value (0 - 65535)', fontsize=11, labelpad=10)
        plt.ylabel('Frequency', fontsize=11, labelpad=10)
        plt.grid(axis='y', linestyle=':', alpha=0.6)

        plt.gca().spines['top'].set_visible(False)
        plt.gca().spines['right'].set_visible(False)
        plt.tight_layout()
        plt.show()
    else:
        print("Intensity data not found in this file.")

# Run the updated visualizer
plot_professional_histograms(file_path)

In [ ]:
import laspy
import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import griddata

file_path = '/content/sample_data/NN7232_2PPM_LAS_PHASE1.laz'

def create_interactive_dtm(path, resolution=5):
    # 1. Load the data
    las = laspy.read(path)

    # 2. Filter for Ground Points (Class 2)
    ground_mask = las.classification == 2
    gx = las.x[ground_mask]
    gy = las.y[ground_mask]
    gz = las.z[ground_mask]

    print(f"Processing {len(gx):,} ground points...")

    # 3. Define the Grid
    # resolution=5 means one elevation value every 5 units (meters)
    x_range = np.arange(gx.min(), gx.max(), resolution)
    y_range = np.arange(gy.min(), gy.max(), resolution)
    grid_x, grid_y = np.meshgrid(x_range, y_range)

    # 4. Interpolate (Linear interpolation creates a smooth but realistic surface)
    print("Interpolating surface (this may take a few seconds)...")
    grid_z = griddata((gx, gy), gz, (grid_x, grid_y), method='linear')

    # 5. Create Interactive 3D Surface
    fig = go.Figure(data=[go.Surface(
        x=x_range,
        y=y_range,
        z=grid_z,
        colorscale='earth',
        colorbar=dict(title="Elevation (Z)")
    )])

    # 6. Professional Layout
    fig.update_layout(
        title='Interactive Digital Terrain Model (DTM) - Bare Earth',
        scene=dict(
            xaxis_title='X (East)',
            yaxis_title='Y (North)',
            zaxis_title='Elevation',
            aspectmode='data' # Keeps the terrain height proportional to width
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        template="plotly_dark"
    )

    fig.show()

# Run the DTM generator
# Increase resolution (e.g., to 10) if your computer is slow
create_interactive_dtm(file_path, resolution=5)